# Loader — `nyc_rent_panel.csv`

One row = **one StreetEasy neighborhood in one month**. 176 neighborhoods × 201 months (2010-01 → 2026-09) = 35,376 rows × 29 columns.
Built by `nyc_rent_exploration.ipynb` (Part D). No target columns yet — every column uses only data up to that month.

In [1]:
import pandas as pd
df = pd.read_csv("nyc_rent_panel.csv", parse_dates=["date"])
print(df.shape)
df.head()

(35376, 29)


,areaName,Borough,date,merge_group,shares_nta,area_km2,med_inventory,in_model,rent,inventory,...,net_units,pipeline_units,evictions,subway_complexes,new_units_12m_km2,pipeline_km2,evictions_12m_km2,subway_km2,covid_period,eviction_moratorium
0,Astoria,Queens,2010-01-01,Astoria,0,10.825086,1306.0,1,1600.0,99.0,...,25.0,1933.0,NaN,7.0,0.0,178.566709,NaN,0.646646,0,0
1,Astoria,Queens,2010-02-01,Astoria,0,10.825086,1306.0,1,1650.0,98.0,...,79.0,1876.0,NaN,7.0,0.0,173.301162,NaN,0.646646,0,0
2,Astoria,Queens,2010-03-01,Astoria,0,10.825086,1306.0,1,1600.0,106.0,...,90.0,1802.0,NaN,7.0,0.0,166.465189,NaN,0.646646,0,0
3,Astoria,Queens,2010-04-01,Astoria,0,10.825086,1306.0,1,1600.0,78.0,...,123.0,1854.0,NaN,7.0,0.0,171.268846,NaN,0.646646,0,0
4,Astoria,Queens,2010-05-01,Astoria,0,10.825086,1306.0,1,1662.0,57.0,...,119.0,1856.0,NaN,7.0,0.0,171.453602,NaN,0.646646,0,0


## Columns

| Group | Column | Meaning |
|---|---|---|
| Keys | `areaName`, `Borough`, `date` | StreetEasy neighborhood, borough, month (1st of month) |
| Mapping | `merge_group` | Neighborhoods that share an official NTA and could be merged later |
| | `shares_nta` | 1 = shares an NTA with another neighborhood (merge candidate), 0 = clean link |
| | `area_km2` | Area of the NTA(s) the neighborhood maps to |
| Filter | `med_inventory` | Median monthly rental listings since 2015 |
| | `in_model` | 1 if `med_inventory` ≥ 20 (100 neighborhoods) — smaller ones are too noisy |
| StreetEasy | `rent` | Median asking rent ($/month) |
| | `inventory` | Number of rental listings |
| | `rent_cut_share` | Share of rental listings with a price cut |
| | `sale_price` | Median asking sale price ($) |
| | `sale_cut_share` | Share of sale listings with a price cut |
| StreetEasy, derived | `rent_3m` | 3-month rolling mean of `rent` |
| | `past_12m` | % change in `rent_3m` over the last 12 months |
| | `inventory_yoy`, `sale_price_yoy` | % change vs. 12 months earlier |
| | `price_to_rent` | `sale_price` / (12 × `rent`) — years of rent to buy |
| City (monthly counts) | `new_units` | Units in new buildings completed that month (Housing Database) |
| | `net_units` | Net change in units: new + alterations − demolitions |
| | `pipeline_units` | Units permitted but not yet completed, as of that month |
| | `evictions` | Residential evictions executed that month (from 2017) |
| | `subway_complexes` | Subway station complexes in the area (static) |
| City (density) | `new_units_12m_km2` | New units completed in the last 12 months, per km² |
| | `pipeline_km2`, `subway_km2` | Pipeline units / stations per km² |
| | `evictions_12m_km2` | Evictions in the last 12 months per km² (from 2017-12) |
| Flags | `covid_period` | 1 for 2020-03 → 2021-06 |
| | `eviction_moratorium` | 1 for 2020-03 → 2022-01 |

## Things to know
- **Missing ≠ zero.** `rent` is empty when a neighborhood had no listings that month; `evictions` is empty before 2017.
- **Neighborhoods sharing an NTA** (`shares_nta = 1`) get identical city features — the data is only available for the whole NTA.
- **Use `in_model == 1`** for analysis; the rest are kept so thresholds can change later.

In [2]:
model = df[df.in_model == 1]
print(model.areaName.nunique(), "neighborhoods |", model.date.min().date(), "->", model.date.max().date())
model.drop_duplicates("areaName").Borough.value_counts()

100 neighborhoods | 2010-01-01 -> 2026-09-01


Borough
Brooklyn     40
Manhattan    30
Queens       21
Bronx         9
Name: count, dtype: int64

In [3]:
df.isna().mean().round(3).rename("share_missing").to_frame().T

,areaName,Borough,date,merge_group,shares_nta,area_km2,med_inventory,in_model,rent,inventory,...,net_units,pipeline_units,evictions,subway_complexes,new_units_12m_km2,pipeline_km2,evictions_12m_km2,subway_km2,covid_period,eviction_moratorium
share_missing,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.108,0.0,...,0.0,0.0,0.418,0.0,0.0,0.0,0.473,0.0,0.0,0.0


In [4]:
model.describe().T.round(2)

,count,mean,min,25%,50%,75%,max,std
date,20100,2018-05-01 22:05:22.388059648,2010-01-01 00:00:00,2014-03-01 00:00:00,2018-05-01 00:00:00,2022-07-01 00:00:00,2026-09-01 00:00:00,NaN
shares_nta,20100.0,0.32,0.0,0.0,0.0,1.0,1.0,0.466488
area_km2,20100.0,3.553731,0.517986,1.600003,2.627066,4.821971,16.742078,2.737605
med_inventory,20100.0,336.45,23.0,61.0,142.5,322.0,2651.0,492.821987
in_model,20100.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0
rent,19931.0,2817.875521,550.0,1950.0,2500.0,3345.0,15000.0,1278.088134
inventory,20100.0,308.40199,0.0,40.0,112.0,307.0,6110.0,532.683702
rent_cut_share,19931.0,0.136275,0.0,0.0824,0.128205,0.179712,1.0,0.087951
sale_price,19827.0,1075647.161396,64999.5,485000.0,849000.0,1349000.0,19500000.0,899282.416083
sale_cut_share,19827.0,0.086176,0.0,0.045455,0.081272,0.117647,1.0,0.066286
